# Phase 2E Router 학습

이 notebook은 전처리된 Phase 2E Router JSONL만 읽어 Legacy/Semantic Softmax Router를 학습합니다. 정적 분석과 LLM API 호출은 수행하지 않습니다.

데이터가 없다면 PowerShell에서 먼저 다음 명령을 한 번 실행하세요.

```powershell
python -m llm_security.cli phase2e-prepare `
  --cases data\arvo\cases_all.jsonl `
  --data-dir data\phase2e `
  --seed 2026
```

In [1]:
import gc
import json
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

DATA_DIR = ROOT / 'data' / 'phase2e'
ARTIFACT_DIR = ROOT / 'artifacts' / 'phase2e'
LEGACY_MODEL_PATH = ARTIFACT_DIR / 'router_legacy_v1.pkl'
SEMANTIC_MODEL_PATH = ARTIFACT_DIR / 'router_semantic_v1.pkl'
SUMMARY_PATH = ARTIFACT_DIR / 'router_training_summary.json'
SEED = 2026
TARGET_COVERAGE = 0.95
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

from llm_security.datasets import load_router_samples_jsonl
from llm_security.models import to_dict
from llm_security.routing import (
    AdaptiveExpertRouter,
    RoutingPolicyConfig,
)

## 1. 전처리 데이터 확인

학습 데이터는 backend별로 순차 로드하므로 Legacy와 Semantic 원본을 동시에 메모리에 보관하지 않습니다.

In [2]:
required_files = [
    DATA_DIR / 'split_manifest.json',
    DATA_DIR / 'legacy' / 'router_train.jsonl',
    DATA_DIR / 'semantic' / 'router_train.jsonl',
    DATA_DIR / 'semantic' / 'router_dev.jsonl',
]
missing = [path for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Phase 2E 전처리 파일이 없습니다. phase2e-prepare를 먼저 실행하세요:\n'
        + '\n'.join(str(path) for path in missing)
    )

split_manifest = json.loads(
    (DATA_DIR / 'split_manifest.json').read_text(encoding='utf-8')
)
print('seed:', split_manifest['seed'])
for split, details in split_manifest['splits'].items():
    print(split, 'projects=', details['project_count'], 'cases=', details['case_count'])

seed: 2026
dev projects= 36 cases= 485
test projects= 36 cases= 735
train projects= 165 cases= 2940


## 2. Legacy Router 학습 및 저장

In [3]:
legacy_train = load_router_samples_jsonl(
    DATA_DIR / 'legacy' / 'router_train.jsonl'
)
legacy_family_counts = Counter(
    sample.labels[0].value for sample in legacy_train if len(sample.labels) == 1
)
legacy_router = AdaptiveExpertRouter.fit(
    legacy_train,
    policy_config=RoutingPolicyConfig(),
    seed=SEED,
    use_rule_fallback=True,
)
legacy_router.save(LEGACY_MODEL_PATH)
legacy_summary = {
    'model_path': str(LEGACY_MODEL_PATH),
    'train_samples': len(legacy_train),
    'family_distribution': dict(sorted(legacy_family_counts.items())),
    'learned_families': [family.value for family in legacy_router.available_families],
    'feature_schema': legacy_router.feature_schema_version,
}
pprint(legacy_summary)
del legacy_train, legacy_router
gc.collect()

{'family_distribution': {'control_state_error': 823,
                         'lifetime_resource': 183,
                         'memory_bounds': 1358},
 'feature_schema': 'legacy-v1',
 'learned_families': ['control_state_error',
                      'lifetime_resource',
                      'memory_bounds'],
 'model_path': 'C:\\Users\\junhyun111\\Desktop\\llm-security\\artifacts\\phase2e\\router_legacy_v1.pkl',
 'train_samples': 2364}


20

## 3. Semantic Router 학습, dev calibration 및 저장

Adaptive Top-K threshold는 test가 아닌 dev split으로만 보정합니다.

In [4]:
semantic_train = load_router_samples_jsonl(
    DATA_DIR / 'semantic' / 'router_train.jsonl'
)
semantic_family_counts = Counter(
    sample.labels[0].value for sample in semantic_train if len(sample.labels) == 1
)
semantic_router = AdaptiveExpertRouter.fit(
    semantic_train,
    policy_config=RoutingPolicyConfig(),
    seed=SEED,
    use_rule_fallback=True,
)
del semantic_train
gc.collect()

semantic_dev = load_router_samples_jsonl(
    DATA_DIR / 'semantic' / 'router_dev.jsonl'
)
supported = set(semantic_router.available_families)
semantic_dev_supported = [
    sample
    for sample in semantic_dev
    if len(sample.labels) == 1 and sample.labels[0] in supported
]
if not semantic_dev_supported:
    raise ValueError('Semantic dev split에 학습된 family의 sample이 없습니다.')
semantic_router.triggers.enabled = False
calibration = semantic_router.calibrate_policy(
    semantic_dev_supported, target_coverage=TARGET_COVERAGE
)
semantic_router.triggers.enabled = True
semantic_router.save(SEMANTIC_MODEL_PATH)
semantic_summary = {
    'model_path': str(SEMANTIC_MODEL_PATH),
    'train_samples': sum(semantic_family_counts.values()),
    'dev_calibration_samples': len(semantic_dev_supported),
    'family_distribution': dict(sorted(semantic_family_counts.items())),
    'learned_families': [family.value for family in semantic_router.available_families],
    'feature_schema': semantic_router.feature_schema_version,
    'policy_calibration': to_dict(calibration),
}
pprint(semantic_summary)
del semantic_dev, semantic_dev_supported, semantic_router
gc.collect()

{'dev_calibration_samples': 413,
 'family_distribution': {'control_state_error': 884,
                         'lifetime_resource': 233,
                         'memory_bounds': 1446},
 'feature_schema': 'semantic-v1',
 'learned_families': ['control_state_error',
                      'lifetime_resource',
                      'memory_bounds'],
 'model_path': 'C:\\Users\\junhyun111\\Desktop\\llm-security\\artifacts\\phase2e\\router_semantic_v1.pkl',
 'policy_calibration': {'achieved_coverage': 0.7288135593220338,
                        'average_experts_per_candidate': 1.9757869249394673,
                        'high_confidence': 0.85,
                        'min_margin': 0.05,
                        'target_coverage': 0.95,
                        'target_met': False},
 'train_samples': 2563}


0

## 4. 학습 결과 확인

In [5]:
summary = {
    'seed': SEED,
    'target_coverage': TARGET_COVERAGE,
    'llm_api_calls': 0,
    'legacy': legacy_summary,
    'semantic': semantic_summary,
}
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)

legacy_check = AdaptiveExpertRouter.load(LEGACY_MODEL_PATH)
semantic_check = AdaptiveExpertRouter.load(SEMANTIC_MODEL_PATH)
print('Legacy model:', LEGACY_MODEL_PATH)
print('Semantic model:', SEMANTIC_MODEL_PATH)
print('Training summary:', SUMMARY_PATH)
print('artifact validation: PASS')

Legacy model: C:\Users\junhyun111\Desktop\llm-security\artifacts\phase2e\router_legacy_v1.pkl
Semantic model: C:\Users\junhyun111\Desktop\llm-security\artifacts\phase2e\router_semantic_v1.pkl
Training summary: C:\Users\junhyun111\Desktop\llm-security\artifacts\phase2e\router_training_summary.json
artifact validation: PASS
